# Full atmosphere testing

This notebook tests the backend setup for an atmosphere with molecular and aerosol components.

In [ ]:
import eradiate
import numpy as np
import seaborn as sns
from eradiate.contexts import KernelContext
from eradiate.experiments import AtmosphereExperiment
from eradiate.units import unit_registry as ureg
from util import Result, reshape_pplane

import eradiate_disort as ed

eradiate.fresolver.prepend("data")
eradiate.set_mode("ckd")
sns.set_theme(style="ticks")

SPP = 100_000
if eradiate.get_mode().is_ckd:
    SPP //= 16


def full_atmo(
    sza: float = 30.0,
    has_scattering: bool = True,
    has_absorption: bool = True,
    surface_reflectance: float = 0.0,
):
    return AtmosphereExperiment(
        geometry={
            "type": "plane_parallel",
            "toa_altitude": 100.0 * ureg.km,
            "zgrid": np.linspace(0, 100, 101) * ureg.km,
        },
        surface={"type": "lambertian", "reflectance": surface_reflectance},
        atmosphere={
            "type": "heterogeneous",
            "molecular_atmosphere": {
                "has_scattering": has_scattering,
                "has_absorption": has_absorption,
            },
            "particle_layers": {
                "has_scattering": has_scattering,
                "has_absorption": has_absorption,
                "tau_ref": 0.2,
                "particle_properties": "soot.mie-aer_core_v2",
            },
        },
        illumination={"type": "directional", "zenith": sza, "azimuth": 0.0},
        measures={
            "type": "mdistant",
            "construct": "hplane",
            "azimuth": 0.0,
            "zeniths": np.arange(-75.0, 76.0, 1.0),
            "srf": {"type": "delta", "wavelengths": [550.0]},
        },
    )


exp = full_atmo(sza=30.0)
ctx = KernelContext()
exp.atmosphere.eval_radprops(ctx.si, optional_fields=True)

In [ ]:
results = {}
CASES = {
    "scattering": {
        "has_absorption": False,
        "has_scattering": True,
        "surface_reflectance": 0.0,
    },
    "absorption": {
        "has_absorption": True,
        "has_scattering": False,
        "surface_reflectance": 1.0,
    },
    "full_black": {
        "has_absorption": True,
        "has_scattering": True,
        "surface_reflectance": 0.0,
    },
    "full_white": {
        "has_absorption": True,
        "has_scattering": True,
        "surface_reflectance": 1.0,
    },
}

for case_id, kwargs in CASES.items():
    print(f"Processing case {case_id!r}")
    if case_id in results:
        continue

    exp = full_atmo(**kwargs)
    result = Result()

    result.mitsuba = eradiate.run(exp, spp=SPP)["radiance"].squeeze()
    backend = ed.EradiateDisortBackend()
    result.disort = reshape_pplane(backend.run(exp))

    results[case_id] = result